### ============================================================
## CARDIOVASCULAR DISEASE PREDICTION
## Notebook 1: Data Preprocessing & EDA
## MSc IT with Data Analytics — UWS 2025-26
### ============================================================

# Import

In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Display settings

In [40]:
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')

# --- Load Dataset ---

In [41]:
df = pd.read_csv('../data/cardio_train.csv', sep=';')

# --- Initial Exploration ---
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
print(df.describe())

Dataset Shape: (70000, 13)

First 5 rows:
   id    age  gender  height  weight  ap_hi  ap_lo  cholesterol  gluc  smoke  \
0   0  18393       2     168    62.0    110     80            1     1      0   
1   1  20228       1     156    85.0    140     90            3     1      0   
2   2  18857       1     165    64.0    130     70            3     1      0   
3   3  17623       2     169    82.0    150    100            1     1      0   
4   4  17474       1     156    56.0    100     60            1     1      0   

   alco  active  cardio  
0     0       1       0  
1     0       1       1  
2     0       0       1  
3     0       1       1  
4     0       0       0  

Data Types:
id               int64
age              int64
gender           int64
height           int64
weight         float64
ap_hi            int64
ap_lo            int64
cholesterol      int64
gluc             int64
smoke            int64
alco             int64
active           int64
cardio           int64
dtype: ob

## Drop ID column

In [42]:
# --- Drop ID Column ---
df = df.drop('id', axis=1)
print("ID column dropped. New shape:", df.shape)

ID column dropped. New shape: (70000, 12)


## Step 2 — Convert age from days to years:

In [43]:
# --- Convert Age from Days to Years ---
df['age'] = (df['age'] / 365).round(1)

print("Age conversion complete")
print(f"Age range: {df['age'].min()} to {df['age'].max()} years")
print(f"Mean age: {df['age'].mean():.1f} years")

Age conversion complete
Age range: 29.6 to 65.0 years
Mean age: 53.3 years


### Step 3 — Calculate BMI: BMI = weight (kg) / height (m)²

In [44]:
# --- Calculate BMI ---
df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)

print("BMI column created")
print(f"BMI range: {df['bmi'].min():.1f} to {df['bmi'].max():.1f}")
print(f"Mean BMI: {df['bmi'].mean():.1f}")
print(f"\nBMI Statistics:")
print(df['bmi'].describe())

BMI column created
BMI range: 3.5 to 298.7
Mean BMI: 27.6

BMI Statistics:
count    70000.000000
mean        27.556513
std          6.091511
min          3.471784
25%         23.875115
50%         26.374068
75%         30.222222
max        298.666667
Name: bmi, dtype: float64


### Step 4 — Remove Outliers

In [45]:
# --- Remove Outliers Using Clinical Reference Ranges ---

print("Shape before outlier removal:", df.shape)

# Blood pressure - clinically valid ranges
df = df[df['ap_hi'] >= 70]
df = df[df['ap_hi'] <= 250]
df = df[df['ap_lo'] >= 40]
df = df[df['ap_lo'] <= 150]

# ap_lo must always be less than ap_hi
df = df[df['ap_lo'] < df['ap_hi']]

# Height - realistic adult range in cm
df = df[df['height'] >= 140]
df = df[df['height'] <= 210]

# Weight - realistic adult range in kg
df = df[df['weight'] >= 40]
df = df[df['weight'] <= 180]

# BMI - clinically valid range
df = df[df['bmi'] >= 10]
df = df[df['bmi'] <= 60]

print("Shape after outlier removal:", df.shape)
print(f"Rows removed: {70000 - len(df)}")

Shape before outlier removal: (70000, 13)
Shape after outlier removal: (68455, 13)
Rows removed: 1545


### Step 5 — Remove Impossible Blood Pressure Values

In [46]:
# --- Verify Blood Pressure Logic ---
print("Checking blood pressure consistency...")
print(f"Cases where ap_lo >= ap_hi: {len(df[df['ap_lo'] >= df['ap_hi']])}")

# Check clean ranges
print(f"\nap_hi range: {df['ap_hi'].min()} to {df['ap_hi'].max()}")
print(f"ap_lo range: {df['ap_lo'].min()} to {df['ap_lo'].max()}")
print(f"BMI range: {df['bmi'].min():.1f} to {df['bmi'].max():.1f}")
print(f"Height range: {df['height'].min()} to {df['height'].max()}")
print(f"Weight range: {df['weight'].min()} to {df['weight'].max()}")

print("\nShape confirmed:", df.shape)

Checking blood pressure consistency...
Cases where ap_lo >= ap_hi: 0

ap_hi range: 70 to 240
ap_lo range: 40 to 150
BMI range: 13.5 to 59.5
Height range: 140 to 207
Weight range: 40.0 to 180.0

Shape confirmed: (68455, 13)


### Step 6 — Check Class Balance

In [47]:
# --- Check Class Balance ---
print("Target Variable Distribution:")
print(df['cardio'].value_counts())
print("\nPercentage:")
print(df['cardio'].value_counts(normalize=True) * 100)

Target Variable Distribution:
cardio
0    34581
1    33874
Name: count, dtype: int64

Percentage:
cardio
0    50.516398
1    49.483602
Name: proportion, dtype: float64


### Step 7 — Feature Engineering & Categorisation

In [48]:
# --- BMI Category ---
def bmi_category(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

df['bmi_category'] = df['bmi'].apply(bmi_category)

# --- Age Category ---
def age_category(age):
    if age < 40:
        return 'Under 40'
    elif age < 50:
        return '40-49'
    elif age < 60:
        return '50-59'
    else:
        return '60+'

df['age_category'] = df['age'].apply(age_category)

print("New features created!")
print("\nBMI Category Distribution:")
print(df['bmi_category'].value_counts())
print("\nAge Category Distribution:")
print(df['age_category'].value_counts())

New features created!

BMI Category Distribution:
bmi_category
Normal         25415
Overweight     24609
Obese          17833
Underweight      598
Name: count, dtype: int64

Age Category Distribution:
age_category
50-59       34703
40-49       19086
60+         13093
Under 40     1573
Name: count, dtype: int64


### Save cleaned data

In [49]:
# --- Final Check Before Export ---
print("Final dataset info:")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nMissing values: {df.isnull().sum().sum()}")

# --- Export Clean Dataset ---
df.to_csv('../data/cardio_clean.csv', index=False)
print("\nClean dataset exported successfully to data/cardio_clean.csv!")

Final dataset info:
Shape: (68455, 15)

Columns: ['age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio', 'bmi', 'bmi_category', 'age_category']

Missing values: 0

Clean dataset exported successfully to data/cardio_clean.csv!
